# Part 0 — VMRD dataset and detection tensors

Object detection is a fundamental task in computer vision. While image classification asks *what* is in an image, detection asks *what* is there and *where*. For each input image, the model predicts object classes and bounding boxes. In this assignment, we will work with the Visual Manipulation Relationship Dataset (VMRD), which contains cluttered tabletop scenes collected for robotic grasping.

In this notebook you will:

1. extract the included dataset,
2. understand how a fixed-size grid can represent a variable number of objects,
3. get familiar with the VMRD dataset and its annotations, and
4. build the Dataset and DataLoaders.

In [ ]:
%load_ext autoreload
%autoreload 2

import glob
import json
import os
import tarfile
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
from PIL import Image
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

import assignment2 as a2

DATASET_ROOT = "./data/VMRD_COCO"
print(f"PyTorch: {torch.__version__}")

## 1. Extract the dataset

The dataset is included as several compressed archive parts so that it can be distributed with the assignment repository. Run the following cell once. It combines the parts and extracts the dataset to `data/VMRD_COCO/`. If that folder already exists, the cell leaves it unchanged.

In [ ]:
annotation_file = os.path.join(
    DATASET_ROOT, "annotations", "instances_train.json"
)

if not os.path.exists(annotation_file):
    archive_parts = sorted(glob.glob("./data/VMRD_COCO.tar.gz.part-*"))
    if not archive_parts:
        raise FileNotFoundError("The VMRD_COCO archive parts are missing")

    archive_path = "./data/VMRD_COCO.tar.gz"
    with open(archive_path, "wb") as combined_archive:
        for part_path in archive_parts:
            with open(part_path, "rb") as archive_part:
                combined_archive.write(archive_part.read())

    try:
        with tarfile.open(archive_path, "r:gz") as archive:
            archive.extractall("./data", filter="data")
    finally:
        os.remove(archive_path)

print("Dataset ready at:", DATASET_ROOT)

## 2. Get familiar with the dataset

We provide the VMRD dataset in the COCO format, which is a common format for object detection tasks (because one of the most popular object detection datasets, MS COCO, uses this format). The dataset is split into training, validation, and test sets. 

In [ ]:
# dataset sizes
train_len = len(os.listdir(os.path.join(DATASET_ROOT, "images", "train")))
val_len = len(os.listdir(os.path.join(DATASET_ROOT, "images", "val")))
test_len = len(os.listdir(os.path.join(DATASET_ROOT, "images", "test")))

print(f"Training set size: {train_len}")
print(f"Validation set size: {val_len}")
print(f"Test set size: {test_len}")

The annotations are stored in JSON files. Let's load the training annotations and inspect the structure.

It roughly has the following structure:

- `info`: general dataset information
- `licenses`: optional licensing information
- `images`: one record per image, for each image:
    - `id`: unique image ID
    - `file_name`: relative path to the image file
    - `width`, `height`: image dimensions in pixels
- `annotations`: one record per object
    - `id`: unique annotation ID
    - `image_id`: links to an image
    - `category_id`: links to a category
    - `bbox`: bounding box in `[x, y, width, height]` format, where `(x, y)` is the top-left corner of the box
    - `area`: area of the bounding box
    - `iscrowd`: indicates if the object is a crowd
    - `segmentation`: polygon segmentation of the object, empty for this dataset


In [ ]:
train_annotations = json.load(open(os.path.join(DATASET_ROOT, "annotations", "instances_train.json")))
# the annotations are stored in a dictionary containing "info", "licenses", "images", "annotations", and "categories" keys
print("Top-level keys:", train_annotations.keys())

# image record, it contains the image id, file name, and image size
print("\nFirst image record:")
print(train_annotations["images"][0])

# annotations, containing bounding box information.
print("\nFirst annotation record:")
print(train_annotations["annotations"][0])

# categories
print("\nCategories:")
for category in train_annotations["categories"]:
    print(category)

We can visualize one random image and its annotations using the following code snippet.

In [ ]:
# Randomly choose one image record.
example_image_index = np.random.randint(len(train_annotations["images"]))
example_image_id = train_annotations["images"][example_image_index]["id"]

# Find every bounding box that belongs to this image.
example_image_annotations = [
    annotation for annotation in train_annotations["annotations"]
    if annotation["image_id"] == example_image_id
]
category_names = {
    category["id"]: category["name"]
    for category in train_annotations["categories"]
}

# Read image and plot
image_path = os.path.join(
    DATASET_ROOT, "images", "train", train_annotations["images"][example_image_index]["file_name"]
)
image = Image.open(image_path).convert("RGB")

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(image)
for annotation in example_image_annotations:
    x, y, width, height = annotation["bbox"]
    ax.add_patch(
        Rectangle((x, y), width, height, fill=False, color="red", linewidth=2)
    )
    ax.text(x, y, category_names[annotation["category_id"]], color="red")

ax.set_title(f"image_id={example_image_id} | {len(example_image_annotations)} objects")
ax.axis("off")
plt.show()

## 3. Build the PyTorch Dataset & DataLoader

The next step is to build the PyTorch Dataset and DataLoader that can read the images and annotations. 

### Object detection pipeline
Before building the dataloader, we need to figure out what we want the model to predict during training. An object detector has a harder task than classification: one image may contain no objects, while another may contain several. 

Because neural networks and mini-batches are easier to work with when their outputs have fixed shapes, a simple solution is to divide an image into a grid and let every grid cell make one prediction. Specifically, we use a $16\times 16$ grid in this assignment. Each object is assigned to the cell containing the center of its bounding box, even if the box covers several cells. Since one cell stores only one target, two objects whose centers fall in the same cell cannot both be represented; this is one limitation of this simple detector. The prediction targets for each image are:

- objectness targets: `(16, 16)`, storing `1` when an object's center is in the cell and `0` otherwise;
- class targets: `(16, 16)`, storing a class from `0` to `9`. These values are used only where the objectness target is `1`;
- box targets: `(4, 16, 16)`, storing `[tx, ty, w, h]` for the object assigned to each cell. `tx` and `ty` locate the center within that cell, while `w` and `h` are the box size relative to the whole image.

The figure below shows the targets of one training image. The yellow cells contain the box centers: they are the only cells with objectness `1`, and they store the class of their object. The right panel zooms in on the stapler: `tx` and `ty` are measured from the top-left corner of its cell in units of one cell, so they are always in `[0, 1)`, while `w` and `h` can be larger than one cell.

![Grid targets of one training image](./assets/grid_targets.png)

The model predicts 1 objectness score, 10 class scores, and 4 box values for every cell. Objectness tells us whether the cell contains an object; class and box predictions are used only for object cells.

During inference, nearby cells may predict the same object. A post-processing step called **non-maximum suppression (NMS)** keeps the highest-scoring box among strongly overlapping boxes of the same class and removes the duplicates. The following figure illustrates the grid representation and NMS.

![From bounding boxes to grid targets and NMS](./assets/object_detection_grid_nms.png)

For this assignment, we have implemented the `VMRDDataset` for you in [`assignment2/data.py`](assignment2/data.py). Besides reading the image, it collects all annotations for that image and creates `objectness_targets`, `class_targets`, and `box_targets`. Therefore, each sample contains an image with shape `(3, 256, 256)`, objectness targets with shape `(16, 16)`, class targets with shape `(16, 16)`, and box targets with shape `(4, 16, 16)`. You are suggested to read the code (together with the PyTorch documentation) and understand how it works.

**Your task here:**

After reading the code, you are required to complete the get loader functions in [`assignment2/data.py`](assignment2/data.py) (2 marks):
- `get_train_loader()` (0.5 mark)
- `get_val_loader()` (0.5 mark)
- `get_test_loader()` (0.5 mark)

The other 0.5 mark is for calculating and setting the mean and standard deviation of the training set.

You could follow the following steps here:
1. Calculate the mean and standard deviation of the training set in the below cell. Both mean and std should be a list of three values, one for each channel (R, G, B).
2. Use the calculated mean and std values (approximate to 2 decimal places) to define the transformations accordingly in each get loader function. Read the documentation below to decide which transforms you need and in what order. The image size after resizing should be `(256, 256)`. For training, use `ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)` as data augmentation. Validation and test data should not be augmented.

   **Note:** Unlike image classification, do not use geometric augmentation such as flipping, cropping, or rotation here. The transform is applied only to the image, while the grid targets are calculated from the original box coordinates. Changing the image geometry would move the objects without moving their targets. `ColorJitter` only changes the pixel colors, so the targets remain correct.
3. Wrap each transformed dataset with a DataLoader. Use the given batch size and choose the appropriate value for the rest of the parameters. On TALC, `num_workers=4`, `pin_memory=True`, and `persistent_workers=True` can make data loading faster. If multiprocessing causes problems on your own computer, use `num_workers=0` and omit `persistent_workers`.

Useful documentations:
- [Datasets & Dataloaders](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html)
- [DataLoader API](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)
- [Transforms](https://docs.pytorch.org/tutorials/beginner/basics/transforms_tutorial.html)
- [Transforms for image classification](https://docs.pytorch.org/vision/stable/auto_examples/transforms/plot_transforms_getting_started.html#i-just-want-to-do-image-classification)

In [ ]:
# loader for getting stats of the training set
stats_transform = transforms.Compose([
    transforms.Resize((a2.IMAGE_SIZE, a2.IMAGE_SIZE)),
    transforms.ToTensor(),
])
stats_dataset = a2.VMRDDataset(
    DATASET_ROOT, split="train", transform=stats_transform
)
stats_loader = DataLoader(stats_dataset, batch_size=32, shuffle=False)

####################################
# Your implementation starts here. #
pass
# Your implementation ends here. #
##################################

print("training mean:", train_mean.tolist())
print("training std: ", train_std.tolist())

After you have completed the get loader functions, you can run the following code to check the shape of the images and targets. 

In [ ]:
train_loader = a2.get_train_loader(DATASET_ROOT, batch_size=32)
val_loader = a2.get_val_loader(DATASET_ROOT, batch_size=32)
test_loader = a2.get_test_loader(DATASET_ROOT, batch_size=32)

train_images, train_objectness, train_classes, train_boxes = next(iter(train_loader))
print("Training set - Images:", train_images.shape) # expected output: torch.Size([32, 3, 256, 256])
print("Training set - Objectness:", train_objectness.shape) # expected output: torch.Size([32, 16, 16])
print("Training set - Classes:", train_classes.shape) # expected output: torch.Size([32, 16, 16])
print("Training set - Boxes:", train_boxes.shape) # expected output: torch.Size([32, 4, 16, 16])